In [1]:
import pandas as pd

df = pd.read_excel("../data/raw/online_retail_II.xlsx", sheet_name=0)
df["StockCode"] = df["StockCode"].astype(str)
df.head()

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085.0,United Kingdom
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085.0,United Kingdom
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085.0,United Kingdom


In [2]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 525461 entries, 0 to 525460
Data columns (total 8 columns):
 #   Column       Non-Null Count   Dtype         
---  ------       --------------   -----         
 0   Invoice      525461 non-null  object        
 1   StockCode    525461 non-null  str           
 2   Description  522533 non-null  object        
 3   Quantity     525461 non-null  int64         
 4   InvoiceDate  525461 non-null  datetime64[us]
 5   Price        525461 non-null  float64       
 6   Customer ID  417534 non-null  float64       
 7   Country      525461 non-null  str           
dtypes: datetime64[us](1), float64(2), int64(1), object(2), str(2)
memory usage: 32.1+ MB


In [3]:
df.describe()

,Quantity,InvoiceDate,Price,Customer ID
count,525461.000000,525461,525461.000000,417534.000000
mean,10.337667,2010-06-28 11:37:36.845018,4.688834,15360.645478
min,-9600.000000,2009-12-01 07:45:00,-53594.360000,12346.000000
25%,1.000000,2010-03-21 12:20:00,1.250000,13983.000000
50%,3.000000,2010-07-06 09:51:00,2.100000,15311.000000
75%,10.000000,2010-10-15 12:45:00,4.210000,16799.000000
max,19152.000000,2010-12-09 20:01:00,25111.090000,18287.000000
std,107.424110,NaN,146.126914,1680.811316


In [4]:
df.isnull().sum()

Invoice             0
StockCode           0
Description      2928
Quantity            0
InvoiceDate         0
Price               0
Customer ID    107927
Country             0
dtype: int64

In [5]:
print("Przed czyszczeniem:", len(df))

# 1. Usuń wiersze bez Customer ID - nie da się rekomendować anonimowym klientom
df_clean = df.dropna(subset=["Customer ID"])
print("Po usunięciu braków Customer ID:", len(df_clean))

# 2. Usuń zwroty (ujemna ilość = zwrot towaru, nie zakup)
df_clean = df_clean[df_clean["Quantity"] > 0]
print("Po usunięciu zwrotów:", len(df_clean))

# 3. Usuń błędne/zerowe ceny
df_clean = df_clean[df_clean["Price"] > 0]
print("Po usunięciu błędnych cen:", len(df_clean))

# 4. Konwersja Customer ID na int (był float przez obecność NaN, teraz ich nie ma)
df_clean["Customer ID"] = df_clean["Customer ID"].astype(int)

# 5. Reset indeksu (po usuwaniu wierszy zostają "dziury" w numeracji)
df_clean = df_clean.reset_index(drop=True)

# 6. Usuń całkowicie zduplikowane wiersze
df_clean = df_clean.drop_duplicates()
print("Po usunięciu duplikatów:", len(df_clean))

df_clean.info()

Przed czyszczeniem: 525461
Po usunięciu braków Customer ID: 417534
Po usunięciu zwrotów: 407695
Po usunięciu błędnych cen: 407664
Po usunięciu duplikatów: 400916
<class 'pandas.DataFrame'>
Index: 400916 entries, 0 to 407663
Data columns (total 8 columns):
 #   Column       Non-Null Count   Dtype         
---  ------       --------------   -----         
 0   Invoice      400916 non-null  object        
 1   StockCode    400916 non-null  str           
 2   Description  400916 non-null  object        
 3   Quantity     400916 non-null  int64         
 4   InvoiceDate  400916 non-null  datetime64[us]
 5   Price        400916 non-null  float64       
 6   Customer ID  400916 non-null  int64         
 7   Country      400916 non-null  str           
dtypes: datetime64[us](1), float64(1), int64(2), object(2), str(2)
memory usage: 27.5+ MB


In [6]:
customer_product_matrix = df_clean.pivot_table(
    index="Customer ID",
    columns="StockCode",
    values="Quantity",
    aggfunc="sum",
    fill_value=0
)

customer_product_matrix.shape

(4312, 4017)

In [7]:
total_cells = customer_product_matrix.size
zero_cells = (customer_product_matrix == 0).sum().sum()
sparsity = zero_cells / total_cells * 100

print(f"Procent zer w macierzy: {sparsity:.2f}%")

Procent zer w macierzy: 98.42%


In [8]:
from sklearn.metrics.pairwise import cosine_similarity

product_similarity = cosine_similarity(customer_product_matrix.T)

In [9]:
product_similarity_df = pd.DataFrame(
    product_similarity,
    index=customer_product_matrix.columns,
    columns=customer_product_matrix.columns
)

def get_similar_products(product_id: str, n: int = 5) -> pd.Series:
    posortowane = product_similarity_df[product_id].sort_values(ascending=False)
    return posortowane.iloc[1:n + 1]

In [10]:
product_names = (
    df_clean[["StockCode", "Description"]]
    .drop_duplicates(subset="StockCode")
    .set_index("StockCode")
)

recommendations = get_similar_products("10002", n=5)
recommendations_with_names = recommendations.to_frame("similarity_score").join(product_names)

In [11]:
import numpy as np

np.random.seed(42)

test_rows = []

for customer_id in df_clean["Customer ID"].unique():
    customer_data = df_clean[df_clean["Customer ID"] == customer_id]

    if len(customer_data) >= 2:
        test_row = customer_data.sample(n=1, random_state=42)
        test_rows.append(test_row)

test_set = pd.concat(test_rows)
train_set = df_clean.drop(test_set.index)

print("Train set:", len(train_set))
print("Test set:", len(test_set))

Train set: 396695
Test set: 4221


In [12]:
train_matrix = train_set.pivot_table(
    index="Customer ID",
    columns="StockCode",
    values="Quantity",
    aggfunc="sum",
    fill_value=0
)

train_similarity = cosine_similarity(train_matrix.T)

train_similarity_df = pd.DataFrame(
    train_similarity,
    index=train_matrix.columns,
    columns=train_matrix.columns
)

train_matrix.shape

(4312, 4015)

In [13]:
produkty_oryginalne = set(customer_product_matrix.columns)
produkty_treningowe = set(train_matrix.columns)

zniknięte_produkty = produkty_oryginalne - produkty_treningowe
print(zniknięte_produkty)
print(len(zniknięte_produkty))

{'44242A', '20885'}
2


In [14]:
print("44242A:", (df_clean["StockCode"] == "44242A").sum())
print("20885:", (df_clean["StockCode"] == "20885").sum())

44242A: 1
20885: 1


In [15]:
przykladowy_test = test_set.iloc[0]
test_customer = przykladowy_test["Customer ID"]
ukryty_produkt = przykladowy_test["StockCode"]

print("Klient:", test_customer)
print("Ukryty produkt:", ukryty_produkt)

Klient: 13085
Ukryty produkt: 21564


In [16]:
zakupy_klienta_w_train = set(train_set[train_set["Customer ID"] == test_customer]["StockCode"])
print("Produkty klienta w train_set:", zakupy_klienta_w_train)

Produkty klienta w train_set: {'48138', '21564', '22195', '21563', '21199', '85048', '22204', '21523', '22201', '21232', '22350', '21068', '21871', '21198', '22041', '22328', '22326', '79323W', '21137', '22202', '22245', '22349', '22179', '22414', '40046A', '21955', '84992', '22064', '22136', '22353', '22244', '79323P', '22200'}


In [17]:
wszystkie_rekomendacje = set()

for produkt in zakupy_klienta_w_train:
    if produkt in train_similarity_df.columns:
        rekomendacje = train_similarity_df[produkt].sort_values(ascending=False).iloc[1:6]
        wszystkie_rekomendacje.update(rekomendacje.index)

print("Rekomendowane produkty:", wszystkie_rekomendacje)

Rekomendowane produkty: {'35915B', '21154', '21564', '48175', '84858A', '21199', '84805A', '22203', '21368', '22204', '21872', '21558', '22301', '47559B', '22135', '21497', '79323GR', '72023F', '22765', '21198', '22041', '20886', '22305', '48129', '21908', '21331', '79323W', '22196', '22202', '37494A', '72754B', '21107', '79323LP', '21916', '90039B', '22365', '84991', '20719', '22854', '22136', '22360', '84455', '37345', '21735', '90037D', '84925E', '22856', '90037B', '84632', '20913', '20679', '22754', '84375', '84255B', '22243', '22300', '22630', '84029E', '22095', '22027', '22326', '22759', '85116', '22502', '22189', '21955', '21874', '21877', '82580', '21095', '21820', '90037C', '85170A', '22311', '20715', '22188', '84925A', '22131', '22201', '20828', '22350', '22629', '21213', '22613', '35915A', '22646', '37482B', '21367', '20750', '22790', '20952', '22769', '22244', '21171', '21559', '21674', '35971', '48111', '21864', '35609A', '72793', '48188', '21069', '15059A', '21871', '2241

In [18]:
trafienie = ukryty_produkt in wszystkie_rekomendacje
print("Czy trafiono?", trafienie)

Czy trafiono? True


In [19]:
trafienia = 0
total = 0

for _, row in test_set.iterrows():
    test_customer = row["Customer ID"]
    ukryty_produkt = row["StockCode"]

    zakupy_klienta_w_train = set(train_set[train_set["Customer ID"] == test_customer]["StockCode"])

    wszystkie_rekomendacje = set()
    for produkt in zakupy_klienta_w_train:
        if produkt in train_similarity_df.columns:
            rekomendacje = train_similarity_df[produkt].sort_values(ascending=False).iloc[1:6]
            wszystkie_rekomendacje.update(rekomendacje.index)

    if ukryty_produkt in wszystkie_rekomendacje:
        trafienia += 1
    total += 1

precision = trafienia / total
print(f"Trafienia: {trafienia}/{total}")
print(f"Precision: {precision:.2%}")

Trafienia: 1418/4221
Precision: 33.59%


In [20]:
rozmiary_historii = test_set["Customer ID"].apply(
    lambda cid: len(set(train_set[train_set["Customer ID"] == cid]["StockCode"]))
)

print(rozmiary_historii.describe())

count    4221.000000
mean       64.252547
std        86.284192
min         1.000000
25%        17.000000
50%        38.000000
75%        80.000000
max      1741.000000
Name: Customer ID, dtype: float64


In [21]:
wyniki = []

for _, row in test_set.iterrows():
    test_customer = row["Customer ID"]
    ukryty_produkt = row["StockCode"]

    zakupy_klienta_w_train = set(train_set[train_set["Customer ID"] == test_customer]["StockCode"])

    wszystkie_rekomendacje = set()
    for produkt in zakupy_klienta_w_train:
        if produkt in train_similarity_df.columns:
            rekomendacje = train_similarity_df[produkt].sort_values(ascending=False).iloc[1:6]
            wszystkie_rekomendacje.update(rekomendacje.index)

    trafienie = ukryty_produkt in wszystkie_rekomendacje

    wyniki.append({
        "customer_id": test_customer,
        "rozmiar_historii": len(zakupy_klienta_w_train),
        "trafienie": trafienie
    })

wyniki_df = pd.DataFrame(wyniki)

In [24]:
mediana = wyniki_df["rozmiar_historii"].median()

mala_historia = wyniki_df[wyniki_df["rozmiar_historii"] <= mediana]
duza_historia = wyniki_df[wyniki_df["rozmiar_historii"] > mediana]

precision_mala = mala_historia["trafienie"].mean()
precision_duza = duza_historia["trafienie"].mean()

print(f"Klienci z małą historią (≤{mediana:.0f} produktów): {len(mala_historia)} osób, precision: {precision_mala:.2%}")
print(f"Klienci z dużą historią (>{mediana:.0f} produktów): {len(duza_historia)} osób, precision: {precision_duza:.2%}")

Klienci z małą historią (≤38 produktów): 2111 osób, precision: 25.82%
Klienci z dużą historią (>38 produktów): 2110 osób, precision: 41.37%
